# 10 - Fundamentals (take-home)

**ICSSI 2026, Using LLMs via the API**

This supplement covers additional API basics: the shape of the Messages API,
statelessness, system prompts, the `max_tokens` and `stop_reason` fields, streaming, and
error handling.

In [1]:
# bootstrap: works in Google Colab and locally
import os, sys, subprocess

# this notebook lives in take_home/; walk up to the repo root so claude_kit.py,
# .env and tutorial_data/ all resolve no matter which folder the kernel started in
while not os.path.exists("claude_kit.py") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

try:
    import anthropic
except ImportError:
    pip_install("anthropic")
    import anthropic

# find the api key: env var first, then a Colab secret, then a local .env file
if not os.environ.get("ANTHROPIC_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        try:
            from dotenv import load_dotenv
        except ImportError:
            pip_install("python-dotenv")
            from dotenv import load_dotenv
        load_dotenv()

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "No ANTHROPIC_API_KEY found. Running locally, copy .env.example to .env and add your "
    "key. In Colab, add it in the Secrets panel (the key icon), named ANTHROPIC_API_KEY."
)
print("anthropic version:", anthropic.__version__, "| API key loaded: OK")
MODEL = "claude-haiku-4-5"

anthropic version: 0.111.0 | API key loaded: OK


## The shape of the Messages API

A conversation is a list of messages, each with a role and content. The roles are `user`
and `assistant`, and the first message must be `user`. The reply is in `response.content`,
which is a list of blocks.

In [2]:
import anthropic
client = anthropic.Anthropic()

response = client.messages.create(
    model=MODEL, max_tokens=150,
    messages=[{"role": "user", "content": "Name three subfields of the science of science."}],
)
print(response.content[0].text)

# Three Subfields of the Science of Science

1. **Scientometrics** - The quantitative study of science using metrics like citation counts, publication rates, and h-indices to measure research impact and productivity.

2. **Science and Technology Studies (STS)** - The examination of how science and technology develop, interact with society, and are shaped by social, cultural, and political factors.

3. **Research Evaluation and Assessment** - The analysis of research quality, effectiveness, and outcomes, including peer review processes, research funding allocation, and performance measurement.


## The API is stateless, so you hold the memory

A second call knows nothing about the first unless you resend the history. To have a real
conversation, append each turn yourself.
We provide the `claude_kit.Conversation` wrapper class to do this for you.

In [ ]:
history = [{"role": "user", "content": "My research uses patent-to-paper citations."}]
first_response = client.messages.create(model=MODEL, max_tokens=120, messages=history)
history.append(
    {"role": "assistant", "content": first_response.content[0].text}
)  # add response to conversation history
print(f"FIRST RESPONSE: {first_response.content[0].text}")

history.append({"role": "user", "content": "Given that, suggest one analysis I could run."})   
second_response = client.messages.create(model=MODEL, max_tokens=200, messages=history)
print(f"SECOND RESPONSE: {second_response.content[0].text}")   # it remembers the patent-to-paper context

# Suggested Analysis: Citation Lag & Field Maturity

**Compare the time lag between paper publication and subsequent patent citations across research fields.**

## The Idea
- Measure how quickly patents cite papers in different disciplines (e.g., biotech vs. software vs. materials science)
- Shorter lags might indicate fields where academic research quickly becomes applicable
- Longer lags might suggest more fundamental/exploratory research

## Why It's Valuable
- Reveals **technology readiness timelines** by field
- Shows which academic areas have the **most direct commercial relevance**
- Could inform policy decisions about research funding and IP strategy
- Interesting for comparing "curiosity-driven" vs. "applied" research areas

## Quick Implementation
Group your patent-to-paper citations by the paper's field, calculate median time between publication and patent citation, then visualize the distribution.

Would this fit your data and research interests


## System prompts set the role and the rules

The system prompt steers behavior without being part of the back-and-forth. It is good for
instructions like asking the model to act as a careful peer reviewer, setting an output
format, or supplying domain context.

In [4]:
response = client.messages.create(
    model=MODEL, max_tokens=50,
    system="You are a terse methods reviewer. Reply in a single sentence.",
    messages=[{"role": "user", "content": "Critique: 'We prove causality using OLS on observational data.'"}],
)
print(f"METHODS REVIEW: {response.content[0].text}")

METHODS REVIEW: This is problematic because OLS on observational data cannot establish causality without strong unconfoundedness assumptions that are typically unverifiable.


## The max_tokens and stop_reason fields

The `max_tokens` field caps the reply length. If the model hits that cap, `stop_reason`
becomes `max_tokens` and the text is truncated, which is a common gotcha. To check that you're getting the full response, ensure that `stop_reason=end_turn` instead.

In [5]:
response = client.messages.create(
    model=MODEL, max_tokens=16,
    messages=[{"role": "user", "content": "Explain bibliometrics in detail."}],
)
print("stop_reason:", response.stop_reason)
print("text so far:", response.content[0].text)

stop_reason: max_tokens
text so far: # Bibliometrics: A Comprehensive Explanation

## Definition


## Streaming

For long replies or live interfaces, stream tokens as they arrive, which also avoids
timeouts on big outputs. The SDK helper accumulates the final message for you.

In [ ]:
with client.messages.stream(
    model=MODEL, max_tokens=300,
    messages=[{"role": "user", "content": "Write a haiku about citation networks."}],
) as stream:
    for chunk in stream.text_stream:
        print(chunk, end="", flush=True)
    final_message = stream.get_final_message()
print("\n\noutput tokens:", final_message.usage.output_tokens)

## Error handling and retries

The SDK raises typed exceptions and automatically retries rate limits, which are status
429, and server errors, which are status 5xx, with backoff. Catch from the most specific
exception to the least specific.

In [ ]:
try:
    response = client.messages.create(
        model=MODEL, max_tokens=50,
        messages=[{"role": "user", "content": "ping"}],
    )
    print("ok:", response.content[0].text)
except anthropic.RateLimitError:
    print("rate limited; the SDK already retried, so back off and try again later")
except anthropic.APIStatusError as error:
    print("API error:", error.status_code, error.message)
except anthropic.APIConnectionError:
    print("network problem; check your connection")

## Recap

Messages are a list of `user` and `assistant` turns, and the reply is in
`response.content`. The API is stateless, so you resend the history to continue a
conversation. The system prompt steers behavior, and `max_tokens` together with
`stop_reason` governs length. Stream for long output, and rely on typed exceptions and the
built-in retries.